In [0]:

%run /Repos/awproject/databricks_learning/config/config


In [0]:


%run /Repos/awproject/databricks_learning/batch_control/batch_control


In [0]:


# =============================================================================
# CELL 3 — imports
# =============================================================================
from pyspark.sql import functions as F
from pyspark.sql import Window


# =============================================================================
# CELL 4 — helpers
# =============================================================================

def add_audit_cols(df):
    """Drop bronze audit cols and stamp silver audit cols."""
    return (
        df
        .drop("source_file", "ingestion_timestamp", "ingestion_date", "batch_id")
        .withColumn("silver_ingestion_timestamp", F.current_timestamp())
        .withColumn("silver_ingestion_date",      F.current_date())
    )


def apply_dq(df, pk_cols: list, required_cols: list, extra_checks: list = None):
    """
    Tags every row with dq_status (PASS/FAIL) and dq_failed_reason.
    Checks:
      1. Null on required columns
      2. Duplicate on primary key columns
      3. Any extra validity checks passed as list of (condition, reason) tuples
    All rows written to silver — failed rows are tagged not removed.
    """
    # 1. Add duplicate count column first
    df = df.withColumn("_dup_count", F.count("*").over(Window.partitionBy(*pk_cols)))

    # 2. Build reasons expression — _dup_count is now a real column
    reasons = F.lit(None).cast("string")

    # Null checks
    for col in required_cols:
        reasons = F.when(
            F.col(col).isNull(),
            F.concat_ws(" | ", F.coalesce(reasons, F.lit("")), F.lit(f"NULL:{col}"))
        ).otherwise(reasons)

    # Duplicate check — now safe because _dup_count is a real column
    reasons = F.when(
        F.col("_dup_count") > 1,
        F.concat_ws(" | ", F.coalesce(reasons, F.lit("")), F.lit(f"DUPLICATE:{'+'.join(pk_cols)}"))
    ).otherwise(reasons)

    # Extra validity checks
    if extra_checks:
        for condition, reason in extra_checks:
            reasons = F.when(
                condition,
                F.concat_ws(" | ", F.coalesce(reasons, F.lit("")), F.lit(reason))
            ).otherwise(reasons)

    # 3. Apply all reasons in one select, then drop _dup_count
    df = df \
        .withColumn("dq_failed_reason",
            F.when(reasons != "", reasons).otherwise(None)
        ) \
        .withColumn("dq_status",
            F.when(F.col("dq_failed_reason").isNull(), F.lit("PASS"))
             .otherwise(F.lit("FAIL"))
        ) \
        .drop("_dup_count")

    return df


def write_silver(df, table_name: str):
    """Always FULL — overwrite every run."""
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER}.{table_name}")
    )
    total  = df.count()
    passed = df.filter(F.col("dq_status") == "PASS").count()
    failed = total - passed
    print(f"  Rows: {total:,} total | {passed:,} PASS | {failed:,} FAIL")
    return total


# =============================================================================
# CELL 5 — calendar
# Columns : Date
# Format  : M/d/yyyy e.g. 1/1/2015
# PK      : date
# =============================================================================
table_name  = "calendar"
ctrl_bronze = get_last_load(table_name, "bronze")
ctrl_silver = get_last_load(table_name, "silver")

if ctrl_bronze["status"] != "SUCCESS":
    print(f"[BLOCKED] {table_name} — bronze status: {ctrl_bronze['status']}")
elif ctrl_silver["status"] == "SUCCESS":
    print(f"[SKIP] {table_name} — already SUCCESS")
else:
    print(f"[{ctrl_silver['load_type']}] {SILVER}.{table_name}")
    log_batch_start(table_name, "silver", ctrl_silver["load_type"])
    try:
        df = spark.read.table(f"{BRONZE}.calendar")

        # Cast
        df = df.withColumn("Date", F.to_date(F.col("Date"), "M/d/yyyy"))

        # Rename
        df = df.withColumnRenamed("Date", "date")

        # DQ
        df = apply_dq(df,
            pk_cols       = ["date"],
            required_cols = ["date"]
        )

        df   = add_audit_cols(df)
        rows = write_silver(df, table_name)
        log_batch_end(table_name, "silver", None, rows, "SUCCESS")
    except Exception as e:
        log_batch_end(table_name, "silver", None, 0, "FAILED")
        raise


# =============================================================================
# CELL 6 — customers
# Columns : CustomerKey, Prefix, FirstName, LastName, BirthDate, MaritalStatus,
#           Gender, EmailAddress, AnnualIncome, TotalChildren, EducationLevel,
#           Occupation, HomeOwner
# Notes   : AnnualIncome has $ and commas e.g. "$90,000 "
#           BirthDate is M/d/yyyy e.g. 4/8/1966
# PK      : CustomerKey
# =============================================================================
table_name  = "customers"
ctrl_bronze = get_last_load(table_name, "bronze")
ctrl_silver = get_last_load(table_name, "silver")

if ctrl_bronze["status"] != "SUCCESS":
    print(f"[BLOCKED] {table_name} — bronze status: {ctrl_bronze['status']}")
elif ctrl_silver["status"] == "SUCCESS":
    print(f"[SKIP] {table_name} — already SUCCESS")
else:
    print(f"[{ctrl_silver['load_type']}] {SILVER}.{table_name}")
    log_batch_start(table_name, "silver", ctrl_silver["load_type"])
    try:
        df = spark.read.table(f"{BRONZE}.customers")

        # Cast
        df = (df
            .withColumn("CustomerKey",
                F.col("CustomerKey").cast("int"))
            .withColumn("BirthDate",
                F.to_date(F.col("BirthDate"), "M/d/yyyy"))
            .withColumn("AnnualIncome",
                # Strip $, commas, spaces e.g. "$90,000 " → 90000.0
                F.regexp_replace(F.col("AnnualIncome"), r"[$,\s]", "").cast("double"))
            .withColumn("TotalChildren",
                F.col("TotalChildren").cast("int"))
        )

        # Rename
        df = (df
            .withColumnRenamed("CustomerKey",    "customer_key")
            .withColumnRenamed("Prefix",         "prefix")
            .withColumnRenamed("FirstName",      "first_name")
            .withColumnRenamed("LastName",       "last_name")
            .withColumnRenamed("BirthDate",      "birth_date")
            .withColumnRenamed("MaritalStatus",  "marital_status")
            .withColumnRenamed("Gender",         "gender")
            .withColumnRenamed("EmailAddress",   "email_address")
            .withColumnRenamed("AnnualIncome",   "annual_income")
            .withColumnRenamed("TotalChildren",  "total_children")
            .withColumnRenamed("EducationLevel", "education_level")
            .withColumnRenamed("Occupation",     "occupation")
            .withColumnRenamed("HomeOwner",      "home_owner")
        )

        # DQ
        df = apply_dq(df,
            pk_cols       = ["customer_key"],
            required_cols = ["customer_key", "first_name", "last_name", "email_address"],
            extra_checks  = [
                # Annual income must be positive
                (F.col("annual_income") <= 0, "INVALID:annual_income <= 0"),
                # Total children must be non-negative
                (F.col("total_children") < 0,  "INVALID:total_children < 0"),
                # Gender must be M or F
                (~F.col("gender").isin("M", "F"), "INVALID:gender not M or F"),
                # Marital status must be M or S
                (~F.col("marital_status").isin("M", "S"), "INVALID:marital_status not M or S"),
                # HomeOwner must be Y or N
                (~F.col("home_owner").isin("Y", "N"), "INVALID:home_owner not Y or N"),
            ]
        )

        df   = add_audit_cols(df)
        rows = write_silver(df, table_name)
        log_batch_end(table_name, "silver", None, rows, "SUCCESS")
    except Exception as e:
        log_batch_end(table_name, "silver", None, 0, "FAILED")
        raise


# =============================================================================
# CELL 7 — product_categories
# Columns : ProductCategoryKey, CategoryName
# Sample  : 1,Bikes / 2,Components / 3,Clothing / 4,Accessories
# PK      : ProductCategoryKey
# =============================================================================
table_name  = "product_categories"
ctrl_bronze = get_last_load(table_name, "bronze")
ctrl_silver = get_last_load(table_name, "silver")

if ctrl_bronze["status"] != "SUCCESS":
    print(f"[BLOCKED] {table_name} — bronze status: {ctrl_bronze['status']}")
elif ctrl_silver["status"] == "SUCCESS":
    print(f"[SKIP] {table_name} — already SUCCESS")
else:
    print(f"[{ctrl_silver['load_type']}] {SILVER}.{table_name}")
    log_batch_start(table_name, "silver", ctrl_silver["load_type"])
    try:
        df = spark.read.table(f"{BRONZE}.product_categories")

        # Cast
        df = df.withColumn("ProductCategoryKey", F.col("ProductCategoryKey").cast("int"))

        # Rename
        df = (df
            .withColumnRenamed("ProductCategoryKey", "product_category_key")
            .withColumnRenamed("CategoryName",       "category_name")
        )

        # DQ
        df = apply_dq(df,
            pk_cols       = ["product_category_key"],
            required_cols = ["product_category_key", "category_name"]
        )

        df   = add_audit_cols(df)
        rows = write_silver(df, table_name)
        log_batch_end(table_name, "silver", None, rows, "SUCCESS")
    except Exception as e:
        log_batch_end(table_name, "silver", None, 0, "FAILED")
        raise


# =============================================================================
# CELL 8 — product_subcategories
# Columns : ProductSubcategoryKey, SubcategoryName, ProductCategoryKey
# Sample  : 1,Mountain Bikes,1
# PK      : ProductSubcategoryKey
# =============================================================================
table_name  = "product_subcategories"
ctrl_bronze = get_last_load(table_name, "bronze")
ctrl_silver = get_last_load(table_name, "silver")

if ctrl_bronze["status"] != "SUCCESS":
    print(f"[BLOCKED] {table_name} — bronze status: {ctrl_bronze['status']}")
elif ctrl_silver["status"] == "SUCCESS":
    print(f"[SKIP] {table_name} — already SUCCESS")
else:
    print(f"[{ctrl_silver['load_type']}] {SILVER}.{table_name}")
    log_batch_start(table_name, "silver", ctrl_silver["load_type"])
    try:
        df = spark.read.table(f"{BRONZE}.product_subcategories")

        # Cast
        df = (df
            .withColumn("ProductSubcategoryKey", F.col("ProductSubcategoryKey").cast("int"))
            .withColumn("ProductCategoryKey",    F.col("ProductCategoryKey").cast("int"))
        )

        # Rename
        df = (df
            .withColumnRenamed("ProductSubcategoryKey", "product_subcategory_key")
            .withColumnRenamed("SubcategoryName",       "subcategory_name")
            .withColumnRenamed("ProductCategoryKey",    "product_category_key")
        )

        # DQ
        df = apply_dq(df,
            pk_cols       = ["product_subcategory_key"],
            required_cols = ["product_subcategory_key", "subcategory_name", "product_category_key"],
            extra_checks  = [
                # product_category_key must be positive
                (F.col("product_category_key") <= 0, "INVALID:product_category_key <= 0"),
            ]
        )

        df   = add_audit_cols(df)
        rows = write_silver(df, table_name)
        log_batch_end(table_name, "silver", None, rows, "SUCCESS")
    except Exception as e:
        log_batch_end(table_name, "silver", None, 0, "FAILED")
        raise


# =============================================================================
# CELL 9 — products
# Columns : ProductKey, ProductSubcategoryKey, ProductSKU, ProductName,
#           ModelName, ProductDescription, ProductColor, ProductSize,
#           ProductStyle, ProductCost, ProductPrice
# Notes   : ProductCost and ProductPrice are numeric
#           ProductSize and ProductStyle can be 0 (numeric stored as string)
# PK      : ProductKey
# =============================================================================
table_name  = "products"
ctrl_bronze = get_last_load(table_name, "bronze")
ctrl_silver = get_last_load(table_name, "silver")

if ctrl_bronze["status"] != "SUCCESS":
    print(f"[BLOCKED] {table_name} — bronze status: {ctrl_bronze['status']}")
elif ctrl_silver["status"] == "SUCCESS":
    print(f"[SKIP] {table_name} — already SUCCESS")
else:
    print(f"[{ctrl_silver['load_type']}] {SILVER}.{table_name}")
    log_batch_start(table_name, "silver", ctrl_silver["load_type"])
    try:
        df = spark.read.table(f"{BRONZE}.products")

        # Cast
        df = (df
            .withColumn("ProductKey",            F.col("ProductKey").cast("int"))
            .withColumn("ProductSubcategoryKey", F.col("ProductSubcategoryKey").cast("int"))
            .withColumn("ProductCost",           F.col("ProductCost").cast("double"))
            .withColumn("ProductPrice",          F.col("ProductPrice").cast("double"))
        )

        # Rename
        df = (df
            .withColumnRenamed("ProductKey",            "product_key")
            .withColumnRenamed("ProductSubcategoryKey", "product_subcategory_key")
            .withColumnRenamed("ProductSKU",            "product_sku")
            .withColumnRenamed("ProductName",           "product_name")
            .withColumnRenamed("ModelName",             "model_name")
            .withColumnRenamed("ProductDescription",    "product_description")
            .withColumnRenamed("ProductColor",          "product_color")
            .withColumnRenamed("ProductSize",           "product_size")
            .withColumnRenamed("ProductStyle",          "product_style")
            .withColumnRenamed("ProductCost",           "product_cost")
            .withColumnRenamed("ProductPrice",          "product_price")
        )

        # DQ
        df = apply_dq(df,
            pk_cols       = ["product_key"],
            required_cols = ["product_key", "product_name", "product_sku",
                             "product_cost", "product_price"],
            extra_checks  = [
                # Cost must be positive
                (F.col("product_cost") <= 0,
                 "INVALID:product_cost <= 0"),
                # Price must be positive
                (F.col("product_price") <= 0,
                 "INVALID:product_price <= 0"),
                # Cost must not exceed price
                (F.col("product_cost") > F.col("product_price"),
                 "INVALID:product_cost > product_price"),
            ]
        )

        df   = add_audit_cols(df)
        rows = write_silver(df, table_name)
        log_batch_end(table_name, "silver", None, rows, "SUCCESS")
    except Exception as e:
        log_batch_end(table_name, "silver", None, 0, "FAILED")
        raise


# =============================================================================
# CELL 10 — territories
# Columns : SalesTerritoryKey, Region, Country, Continent
# Sample  : 1,Northwest,United States,North America
# PK      : SalesTerritoryKey
# =============================================================================
table_name  = "territories"
ctrl_bronze = get_last_load(table_name, "bronze")
ctrl_silver = get_last_load(table_name, "silver")

if ctrl_bronze["status"] != "SUCCESS":
    print(f"[BLOCKED] {table_name} — bronze status: {ctrl_bronze['status']}")
elif ctrl_silver["status"] == "SUCCESS":
    print(f"[SKIP] {table_name} — already SUCCESS")
else:
    print(f"[{ctrl_silver['load_type']}] {SILVER}.{table_name}")
    log_batch_start(table_name, "silver", ctrl_silver["load_type"])
    try:
        df = spark.read.table(f"{BRONZE}.territories")

        # Cast
        df = df.withColumn("SalesTerritoryKey", F.col("SalesTerritoryKey").cast("int"))

        # Rename
        df = (df
            .withColumnRenamed("SalesTerritoryKey", "sales_territory_key")
            .withColumnRenamed("Region",            "region")
            .withColumnRenamed("Country",           "country")
            .withColumnRenamed("Continent",         "continent")
        )

        # DQ
        df = apply_dq(df,
            pk_cols       = ["sales_territory_key"],
            required_cols = ["sales_territory_key", "region", "country", "continent"]
        )

        df   = add_audit_cols(df)
        rows = write_silver(df, table_name)
        log_batch_end(table_name, "silver", None, rows, "SUCCESS")
    except Exception as e:
        log_batch_end(table_name, "silver", None, 0, "FAILED")
        raise


# =============================================================================
# CELL 11 — sales
# Columns : OrderDate, StockDate, OrderNumber, ProductKey, CustomerKey,
#           TerritoryKey, OrderLineItem, OrderQuantity
# Notes   : OrderDate and StockDate are M/d/yyyy e.g. 1/1/2015
#           StockDate can be older than OrderDate (e.g. 9/21/2001)
# PK      : OrderNumber + OrderLineItem
# =============================================================================
table_name  = "sales"
ctrl_bronze = get_last_load(table_name, "bronze")
ctrl_silver = get_last_load(table_name, "silver")

if ctrl_bronze["status"] != "SUCCESS":
    print(f"[BLOCKED] {table_name} — bronze status: {ctrl_bronze['status']}")
elif ctrl_silver["status"] == "SUCCESS":
    print(f"[SKIP] {table_name} — already SUCCESS")
else:
    print(f"[{ctrl_silver['load_type']}] {SILVER}.{table_name}")
    log_batch_start(table_name, "silver", ctrl_silver["load_type"])
    try:
        df = spark.read.table(f"{BRONZE}.sales")

        # Cast
        df = (df
            .withColumn("OrderDate",     F.to_date(F.col("OrderDate"),  "M/d/yyyy"))
            .withColumn("StockDate",     F.to_date(F.col("StockDate"),  "M/d/yyyy"))
            .withColumn("ProductKey",    F.col("ProductKey").cast("int"))
            .withColumn("CustomerKey",   F.col("CustomerKey").cast("int"))
            .withColumn("TerritoryKey",  F.col("TerritoryKey").cast("int"))
            .withColumn("OrderLineItem", F.col("OrderLineItem").cast("int"))
            .withColumn("OrderQuantity", F.col("OrderQuantity").cast("int"))
        )

        # Rename
        df = (df
            .withColumnRenamed("OrderDate",     "order_date")
            .withColumnRenamed("StockDate",     "stock_date")
            .withColumnRenamed("OrderNumber",   "order_number")
            .withColumnRenamed("ProductKey",    "product_key")
            .withColumnRenamed("CustomerKey",   "customer_key")
            .withColumnRenamed("TerritoryKey",  "territory_key")
            .withColumnRenamed("OrderLineItem", "order_line_item")
            .withColumnRenamed("OrderQuantity", "order_quantity")
        )

        # DQ
        df = apply_dq(df,
            pk_cols       = ["order_number", "order_line_item"],
            required_cols = ["order_date", "order_number", "product_key",
                             "customer_key", "territory_key",
                             "order_line_item", "order_quantity"],
            extra_checks  = [
                # Quantity must be positive
                (F.col("order_quantity") <= 0,
                 "INVALID:order_quantity <= 0"),
                # OrderDate must not be in the future
                (F.col("order_date") > F.current_date(),
                 "INVALID:order_date in future"),
                # All keys must be positive
                (F.col("product_key")   <= 0, "INVALID:product_key <= 0"),
                (F.col("customer_key")  <= 0, "INVALID:customer_key <= 0"),
                (F.col("territory_key") <= 0, "INVALID:territory_key <= 0"),
            ]
        )

        df   = add_audit_cols(df)
        rows = write_silver(df, table_name)
        log_batch_end(table_name, "silver", None, rows, "SUCCESS")
    except Exception as e:
        log_batch_end(table_name, "silver", None, 0, "FAILED")
        raise


# =============================================================================
# CELL 12 — summary
# =============================================================================
show_control_table("silver")

print("\n=== DQ SUMMARY ===")
for t in ["calendar", "customers", "product_categories",
          "product_subcategories", "products", "territories", "sales"]:
    try:
        df     = spark.read.table(f"{SILVER}.{t}")
        total  = df.count()
        passed = df.filter(F.col("dq_status") == "PASS").count()
        failed = total - passed
        pct    = round(passed / total * 100, 1) if total > 0 else 0
        print(f"  {t:<25} {total:>8,} rows | PASS: {passed:,} ({pct}%) | FAIL: {failed:,}")
    except Exception:
        print(f"  {t:<25} table not found")